# Generate recommendations.

## Modules

In [ ]:
import json
import pandas as pd
import re
import os
from IPython.display import clear_output 

from groq import Groq


import rag 
from collaborativeTeachingfiltered import retrieve_strengths
from sentence_transformers import CrossEncoder 
from sentence_transformers import SentenceTransformer
from sentence_transformers import util

from ZFShotRecommendations import format_student_comments
from ZFShotRecommendations import build_recommendation_prompt
from ZFShotRecommendations import generate_recommendation
from ZFShotRecommendations import get_syllabus_data

from MultiAgent import build_mas_teaching_analysis_prompt
from MultiAgent import mas_teaching_analysis
from MultiAgent import build_mas_recommendation_prompt


## Data and models

In [ ]:
df = pd.read_csv("data/comments_df_subsample.csv")

with open("data/labeled - pratices and quotes all courses(50+1)_v2.json", "r") as f:
    result = json.load(f)

df_syllabus = pd.read_csv("data/courses_sample_syllabus.csv")
data_mst = pd.read_csv("data/course_strengths_means.csv")


model = "qwen/qwen3.6-27b"
encoder = SentenceTransformer("intfloat/e5-large-v2")
reranker = CrossEncoder("BAAI/bge-reranker-large")


apk = "your api key"

client = Groq(
    api_key=apk,
)

# Generate Descriptions of Teaching

In [ ]:
strengths = []

prompt = """You are an expert in higher education teaching.
Below are comments written by students in the faculty evaluation survey, alongside an analysis conducted by another model that classifies the presence of teaching practices documented in the literature.

Your task is to generate a description of the STRENGTHS in the teaching strategies found in the material. Focus solely and exclusively on the strengths. Weaknesses are being reviewed separately.

To do this, list all identified strengths along with a description of each and all available details regarding the instructor's specific implementation.
"""

for course in result:
    crs = course['course_id']
    
    tps = ""
    
    for x in course['teaching']:
        tps = f"{tps}\nPractice: {x['practice']}.\nVerbatim quotes: {x['verbatim_quotes']}.\n"
        
    msg = f"""{prompt}
    
    Student comments:
    {format_student_comments(df, crs)}
    
    Classified comments:
    {tps}"""


    chat_completion = client.chat.completions.create(
        messages=[{"role": "user", "content": msg}],
        model = "qwen/qwen3.6-27b",
        temperature=0.9,
        top_p=1,
        max_tokens= 10000
    )

    response = chat_completion.choices[0].message.content
    output = response.split("</think>\n")
    output = output[1]
    
    strengths.append(output)

In [ ]:
with open('data/strengths data.json', "w") as f:
    json.dump(strengths, f, indent = 4)

In [ ]:
stg = strengths.copy()
data_means = df.groupby('course_id')['rating'].mean().reset_index()

data_mst = []

for i, course_strength in enumerate(stg):
    crs = result[i]['course_id']

    course_mean = data_means[data_means['course_id'] == crs]['rating']
    
    d = {
        "course_id2" : crs,
        "description" : course_strength,
        "rating" : course_mean.item()
    }
    
    data_mst.append(d)

data_mst = pd.DataFrame(data_mst)   

data_mst.to_csv("data/course_strengths_means.csv")


In [ ]:
data_mst = pd.read_csv("data/course_strengths_means.csv")

# Generate!

In [ ]:
def Recommend(input_base, sup_info, method, course, client, model, res, df, df_syllabus=None, sources=None, encoder=None, reranker=None, path="", knowledgebase_embeddings=None, knowledge_base=None, strengths=None, data_mst=None, embeddings = None):
    """
    input_base can be one of: 'raw', 'classified' or 'raw+classified'.
    sup_info: 'reviews', 'syllabus', 'knowledge-base', 'syllabus+knowledge-base',
              'syllabus+retrieved-strengths', 'knowledge-base+retrieved-strengths',
              'syllabus+knowledge-base+retrieved-strengths'.
    method can be 'zero-shot', 'multi-agent',
    """
    reflection = None
    srcs = sources

    crs = course

    if method == 'zero-shot':
        if sup_info == 'reviews':
            prompt = build_recommendation_prompt(desc, input_type=input_base)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'syllabus':
            course_objective, program = get_syllabus_data(df_syllabus, crs)
            prompt = build_recommendation_prompt(desc, input_type=input_base, course_objective=course_objective, program=program)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'knowledge-base':
            srcs = sources if sources else rag.get_rag_sources(desc, encoder, knowledgebase_embeddings, knowledge_base, reranker)
            prompt = build_recommendation_prompt(desc, input_type=input_base, quotes=srcs)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'syllabus+knowledge-base':
            course_objective, program = get_syllabus_data(df_syllabus, crs)
            srcs = sources if sources else rag.get_rag_sources(desc, encoder, knowledgebase_embeddings, knowledge_base, reranker)
            prompt = build_recommendation_prompt(desc, input_type=input_base, course_objective=course_objective, program=program, quotes=srcs)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'retrieved-strengths':
            if strengths is None:
                try:
                    strengths = retrieve_strengths(encoder, reranker, data_mst, teaching_description=desc, course_id=crs, embeddings = embeddings, pool_size=3)
                except:
                    strengths = ''
                
            prompt = build_recommendation_prompt(desc, input_type=input_base, other_courses_strengths=strengths)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'syllabus+retrieved-strengths':
            course_objective, program = get_syllabus_data(df_syllabus, crs)
            if strengths is None:
                try:
                    strengths = retrieve_strengths(encoder, reranker, data_mst, teaching_description=desc, course_id=crs, embeddings = embeddings, pool_size=3)
                except:
                    strengths = ''
            prompt = build_recommendation_prompt(desc, input_type=input_base, course_objective=course_objective, program=program, other_courses_strengths=strengths)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'knowledge-base+retrieved-strengths':
            srcs = sources if sources else rag.get_rag_sources(desc, encoder, knowledgebase_embeddings, knowledge_base, reranker)
            if strengths is None:
                try:
                    strengths = retrieve_strengths(encoder, reranker, data_mst, teaching_description=desc, course_id=crs,embeddings = embeddings, pool_size=3)
                except:
                    strengths = ''
                    
            prompt = build_recommendation_prompt(desc, input_type=input_base, quotes=srcs, other_courses_strengths=strengths)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'syllabus+knowledge-base+retrieved-strengths':
            course_objective, program = get_syllabus_data(df_syllabus, crs)
            srcs = sources if sources else rag.get_rag_sources(desc, encoder, knowledgebase_embeddings, knowledge_base, reranker)
            if strengths is None:
                try:
                    strengths = retrieve_strengths(encoder, reranker, data_mst, teaching_description=desc, course_id=crs,embeddings = embeddings, pool_size=3)
                except:
                    strengths = ''
                    
            prompt = build_recommendation_prompt(desc, input_type=input_base, course_objective=course_objective, program=program, quotes=srcs, other_courses_strengths=strengths)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'everything':
            course_objective, program = get_syllabus_data(df_syllabus, crs)
            srcs = sources if sources else rag.get_rag_sources(desc, encoder, knowledgebase_embeddings, knowledge_base, reranker)
            prompt = build_recommendation_prompt(desc, input_type=input_base, course_objective=course_objective, program=program, quotes=srcs)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        else:
            raise ValueError(f"invalid sup_info': {sup_info}")

    elif method == 'multi-agent':
        if sup_info == 'reviews':
            teaching_analysis = mas_teaching_analysis(client, model, desc, input_type=input_base)
            prompt = build_mas_recommendation_prompt(teaching_analysis)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'syllabus':
            course_objective, program = get_syllabus_data(df_syllabus, crs)
            teaching_analysis = mas_teaching_analysis(client, model, desc, input_type=input_base)
            prompt = build_mas_recommendation_prompt(teaching_analysis, course_objective=course_objective, program=program)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'knowledge-base':
            teaching_analysis = mas_teaching_analysis(client, model, desc, input_type=input_base)
            srcs = sources if sources else rag.get_rag_sources(teaching_analysis['instructional_challenges'], encoder, knowledgebase_embeddings, knowledge_base, reranker)
            prompt = build_mas_recommendation_prompt(teaching_analysis, quotes=srcs)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'syllabus+knowledge-base':
            course_objective, program = get_syllabus_data(df_syllabus, crs)
            teaching_analysis = mas_teaching_analysis(client, model, desc, input_type=input_base)
            srcs = sources if sources else rag.get_rag_sources(teaching_analysis['instructional_challenges'], encoder, knowledgebase_embeddings, knowledge_base, reranker)
            prompt = build_mas_recommendation_prompt(teaching_analysis, course_objective=course_objective, program=program, quotes=srcs)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'retrieved-strengths':
            teaching_analysis = mas_teaching_analysis(client, model, desc, input_type=input_base)
            if strengths is None:
                try:
                    strengths = retrieve_strengths(encoder, reranker, data_mst, teaching_description=desc, course_id=crs,embeddings = embeddings, pool_size=3)
                except:
                    strengths = ''
                    
            prompt = build_mas_recommendation_prompt(teaching_analysis, strengths=strengths)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'syllabus+retrieved-strengths':
            course_objective, program = get_syllabus_data(df_syllabus, crs)
            teaching_analysis = mas_teaching_analysis(client, model, desc, input_type=input_base)
            if strengths is None:
                try:
                    strengths = retrieve_strengths(encoder, reranker, data_mst, teaching_description=desc, course_id=crs,embeddings = embeddings, pool_size=3)
                except:
                    strengths = ''
                    
            prompt = build_mas_recommendation_prompt(teaching_analysis, course_objective=course_objective, program=program, strengths=strengths)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'knowledge-base+retrieved-strengths':
            teaching_analysis = mas_teaching_analysis(client, model, desc, input_type=input_base)
            srcs = sources if sources else rag.get_rag_sources(teaching_analysis['instructional_challenges'], encoder, knowledgebase_embeddings, knowledge_base, reranker)
            if strengths is None:
                try:
                    strengths = retrieve_strengths(encoder, reranker, data_mst, teaching_description=desc, course_id=crs,embeddings = embeddings, pool_size=3)
                except:
                    strengths = ''
                    
            prompt = build_mas_recommendation_prompt(teaching_analysis, quotes=srcs, strengths=strengths)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'syllabus+knowledge-base+retrieved-strengths':
            course_objective, program = get_syllabus_data(df_syllabus, crs)
            teaching_analysis = mas_teaching_analysis(client, model, desc, input_type=input_base)
            srcs = sources if sources else rag.get_rag_sources(teaching_analysis['instructional_challenges'], encoder, knowledgebase_embeddings, knowledge_base, reranker)
            if strengths is None:
                try:
                    strengths = retrieve_strengths(encoder, reranker, data_mst, teaching_description=desc, course_id=crs,embeddings = embeddings, pool_size=3)
                except:
                    strengths = ''
                    
            prompt = build_mas_recommendation_prompt(teaching_analysis, course_objective=course_objective, program=program, quotes=srcs, strengths=strengths)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        elif sup_info == 'everything':
            course_objective, program = get_syllabus_data(df_syllabus, crs)
            teaching_analysis = mas_teaching_analysis(client, model, desc, input_type=input_base)
            srcs = sources if sources else rag.get_rag_sources(teaching_analysis['instructional_challenges'], encoder, knowledgebase_embeddings, knowledge_base, reranker)
            prompt = build_mas_recommendation_prompt(teaching_analysis, course_objective=course_objective, program=program, quotes=srcs)
            recommendation = generate_recommendation(client, model, prompt)
            recommendation = re.split(pattern="</think>", string=recommendation)

        else:
            raise ValueError(f"invalid sup_info": {sup_info}")

    else:
        raise ValueError(f"invalid sup_info: {method}")

    return recommendation, reflection, srcs

In [ ]:
path = "scientific literature/"
knowledge_base, knowledgebase_embeddings = rag.semantic_chunking(path, encoder)

In [ ]:
bases = ['raw', 'classified', 'raw+classified']
supinf = ['retrieved-strengths', 'syllabus+retrieved-strengths', 'knowledge-base+retrieved-strengths', 'syllabus+knowledge-base+retrieved-strengths']
methods = ['zero-shot', 'multi-agent']

step = 0

recommendations = []

step_c = 0

In [ ]:
embeddings = encoder.encode(data_mst['description'].tolist())

In [ ]:
for course in result[step_c:]:
    print(f"Step: {len(recommendations) + 1} of {len(result) * len(bases) * len (supinf) * len(methods)}")
    print(f"Generando recomendación para {course['course_id']}")
    crs = course['course_id']
    
    for base in bases:
        print(f"Base: {base}")
        current_sources = None
        
        if base == 'raw':
            desc = format_student_comments(df, crs)
            
        elif base == 'classified':
            desc = str(course)
    
        elif base == 'raw+classified':
            std_com = format_student_comments(df, crs)
            cls_com = str(course)
    
            desc = f"Student comments:\n{str(std_com)}\n\nDetected teaching practices:\n{cls_com}"
        
        else:
            raise ValueError(f"invalid sup_infoo: {input_base}")

        retrieved_strengths = retrieve_strengths(encoder, reranker, data_mst, teaching_description=desc, course_id=crs, pool_size=3)
        
        for sinf in supinf:
            print(f"Considerando: {sinf}")

            if current_sources is None and sinf in ['knowledge-base+retrieved-strengths', 'syllabus+knowledge-base+retrieved-strengths']:
                current_sources = rag.get_rag_sources(
                    desc, 
                    encoder, 
                    knowledgebase_embeddings, 
                    knowledge_base, 
                    reranker)
            
            for method in methods:
                print(f"Método:: {method}")
                
                recommendation = Recommend(
                    input_base = base, 
                    sup_info = sinf, 
                    method = method, 
                    course = crs, 
                    client = client, 
                    model = model, 
                    res = course, 
                    df=df,
                    df_syllabus = df_syllabus, 
                    sources = current_sources, 
                    encoder = encoder, 
                    reranker = reranker, 
                    path = path, 
                    knowledgebase_embeddings = knowledgebase_embeddings, 
                    knowledge_base = knowledge_base,
                    strengths = retrieved_strengths,
                    data_mst = data_mst,
                    embeddings = embeddings)
                
                recommendation = rec_out[0] if isinstance(rec_out, tuple) else rec_out
                
                d = {"course_id" : crs,
                     "base" : base,
                     "supinf" : sinf,
                     "method" : method,
                     "recommendation" : recommendation}
                recommendations.append(d)

                step += 1
                if step != 0 and step % 50 == 0:
                    with open(f'output/recommendations_backup.json', "w", encoding='utf-8') as f:
                        json.dump(recommendations,f, ensure_ascii=False, indent=4) 
                    
    clear_output(wait=True)
    step_c += 1
    
with open("output/recommendation.json", 'w', encoding='utf-8') as f:
    json.dump(recommendations, f, ensure_ascii=False, indent=4)

# Autonomous agent

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

from langchain_core.prompts import ChatPromptTemplate
import pandas as pd

import classifiertoolfoReactAgent
from classifiertoolfoReactAgent import classifier

from reActaGent import getAgentools

In [ ]:
with open("instrument/short_instrument_for_teaching_practices.json", "r") as f:
   instrument = json.load(f)


#######################################################
#
#                    Instrument
#
######################################################


schemas_tp = []

for dim in instrument['Dimensions']:
    for latent_trait in dim['latent_traits']:
        latent_trait['practices']
        
        for p in latent_trait['practices']:
            dct = {
                'practice' : p['practice'],
                'description' : p['description'],
                'step_by_step_analysis' : '',
                'quote' : ''  
            }
            schemas_tp.append(dct)
    
path = "instrument/"            
with open(path + "english_shots.json", "r") as f:
    eng_shots = json.load(f)


 
for p in schemas_tp:
    examples = []
    for s in eng_shots:
        if p['practice'] == s['label']:
            d = dict({
                "text" : s['text'],
                "chain-of-thought" : s['chain-of-thought']
            })
            examples.append(d)
    p['examples'] = examples

schemas_tp = schemas_tp[1:]


for p in schemas_tp:
    examples = []
    for s in eng_shots:
        if p['practice'] == s['label']:
            d = dict({
                "text" : s['text'],
                "chain-of-thought" : s['chain-of-thought']
            })
            examples.append(d)
    p['examples'] = examples

schemas_tp = schemas_tp[1:]

In [ ]:
Model = ChatOpenAI(model = "qwen/qwen3.6-27b",
                                 temperature = 0, 
                                 base_url = "https://api.groq.com/openai/v1", 
                                 api_key = apk,
                                 max_tokens=10000)

herramientas = getAgentools(
    encoder=encoder,
    knowledgebase_embeddings=knowledgebase_embeddings,
    knowledge_base=knowledge_base,
    reranker=reranker,
    df_syllabus=df_syllabus,
    schemas_tp=schemas_tp,
    llm_clasificador=Model,
    data_mst = data_mst,
    
)


In [ ]:
system_prompt = """    
Your task is to identify the foundational problem that acts as a major barrier to student learning and triggers other secondary issues. Then, generate one specific recommendation to address this core challenge, focusing on a solution the instructor can successfully implement with appropriate support.

Rules:
1. Competency Bridge: Your recommendations must use current skills as a foundation to solve a problem within the ZPD.
2. Teaching Style: Your recommendations must strictly respect the teacher's teaching style (what it is centered on and its general orientation).
3. Avoid Overload: Do not suggest changes that require a total course re-engineering or skills that the teacher has not yet demonstrated (i.e., stay within their ZPD).


You are an autonomous agent. You have tools at your disposal. Before giving a final answer, you should think step by step:

1. EVIDENCE EVALUATION: Below you will receive a list of comments written by students in the teacher evaluation survey. If you think you need a deeper understanding of the comments, you can use the 'AnalyzeTeachingPractices' tool to analyze specific comments one by one.
2. CONTEXT: If you need a better understanding of the course context, learning objectives, activities, and assessments, use the 'RequestCourseSyllabus' tool.
3. If you need a deeper understanding of the pedagogical underlying problem and how it could be addressed, use the 'ConsultPedagogicalLiterature' tool.
4. If you want to get inspiration from things other courses do in relation to the identified problems, you can use the 'RetrieveStrengthsfromOtherCourses' tool.
5. SYNTHESIS: Once you have sufficient structured, contextual, and theoretical information, develop your final response.

Required format for the recommendation:


Structure for each Recommendation: 
1. Assign a title, and then write an organic, well-developed text that includes:
2. A brief description of which mentioned teaching failure you are addressing.
3. A description of your proposal to address that problem.
4. A series of concrete and actionable steps to implement these changes in the next version of the course. These must be closely aligned with both the teaching style and the specific context of the course.

You MUST use this json schema for your final answer. In the "recommendation" field, put your final recommendation, and in the "used_tools" field, list the tools you have called to generate this recommendation. 


   {
    "recommendation" : "<your-recommendation>",
    "used_tools" : ["tool x", "tool y", "tool z"]
   }

Omit ```json mark.
PROHIBITED to add absolutely any summary or comment outside of the json schema.
"""

agent_executor = create_react_agent(
    model=Model, 
    tools=herramientas, 
    prompt=system_prompt
)


In [ ]:
autonomous_recommendations = []
idx = 0

In [ ]:
for res in result[idx:]:
    resultado = agent_executor.invoke({
        "messages": [
            ("user", f"{res}")
        ]
    })

    d = {
        'course_id': res['course_id'],
        'method': 'autonomous_agent',
        'recommendation' : resultado["messages"][-1].content
    }

    autonomous_recommendations.append(d)

    idx += 1
    

In [ ]:
with open("output/recommendations_autonomousAgent2_v2.json", 'w', encoding='utf-8') as f:
    json.dump(autonomous_recommendations, f, ensure_ascii=False, indent=4)